In [ ]:
WITH
-- =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =
--复贷系数
-- =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  = 
 cust_level_revolving AS
(
	SELECT  first_loan.user_no
	       ,first_loan.cust_no
	       ,substr(first_loan.loan_time,1,7) AS first_loan_month
	       ,DATE(first_loan.loan_time)       AS first_loan_date
	       ,first_loan.loan_amt              AS first_loan_amt --复贷金额 
	       ,SUM(CASE WHEN DATEDIFF(DATE(loans.loan_time),DATE(first_loan.loan_time)) BETWEEN 0 AND 30 * 1 THEN loans.loan_amt END) reloan_amt1
	       ,SUM(CASE WHEN DATEDIFF(DATE(loans.loan_time),DATE(first_loan.loan_time)) BETWEEN 0 AND 30 * 2 THEN loans.loan_amt END) reloan_amt2
	       ,SUM(CASE WHEN DATEDIFF(DATE(loans.loan_time),DATE(first_loan.loan_time)) BETWEEN 0 AND 30 * 3 THEN loans.loan_amt END) reloan_amt3
	       ,SUM(CASE WHEN DATEDIFF(DATE(loans.loan_time),DATE(first_loan.loan_time)) BETWEEN 0 AND 30 * 4 THEN loans.loan_amt END) reloan_amt4
	       ,SUM(CASE WHEN DATEDIFF(DATE(loans.loan_time),DATE(first_loan.loan_time)) BETWEEN 0 AND 30 * 5 THEN loans.loan_amt END) reloan_amt5
	       ,SUM(CASE WHEN DATEDIFF(DATE(loans.loan_time),DATE(first_loan.loan_time)) BETWEEN 0 AND 30 * 6 THEN loans.loan_amt END) reloan_amt6
	       ,SUM(CASE WHEN DATEDIFF(DATE(loans.loan_time),DATE(first_loan.loan_time)) BETWEEN 0 AND 30 * 7 THEN loans.loan_amt END) reloan_amt7
	       ,SUM(CASE WHEN DATEDIFF(DATE(loans.loan_time),DATE(first_loan.loan_time)) BETWEEN 0 AND 30 * 8 THEN loans.loan_amt END) reloan_amt8
	       ,SUM(CASE WHEN DATEDIFF(DATE(loans.loan_time),DATE(first_loan.loan_time)) BETWEEN 0 AND 30 * 9 THEN loans.loan_amt END) reloan_amt9
	       ,SUM(CASE WHEN DATEDIFF(DATE(loans.loan_time),DATE(first_loan.loan_time)) BETWEEN 0 AND 30 * 10 THEN loans.loan_amt END) reloan_amt10
	       ,SUM(CASE WHEN DATEDIFF(DATE(loans.loan_time),DATE(first_loan.loan_time)) BETWEEN 0 AND 30 * 11 THEN loans.loan_amt END) reloan_amt11
	       ,SUM(CASE WHEN DATEDIFF(DATE(loans.loan_time),DATE(first_loan.loan_time)) BETWEEN 0 AND 30 * 12 THEN loans.loan_amt END) reloan_amt12
	       ,SUM(CASE WHEN DATEDIFF(DATE(loans.loan_time),DATE(first_loan.loan_time)) BETWEEN 0 AND 30 * 13 THEN loans.loan_amt END) reloan_amt13
	       ,SUM(CASE WHEN DATEDIFF(DATE(loans.loan_time),DATE(first_loan.loan_time)) BETWEEN 0 AND 30 * 14 THEN loans.loan_amt END) reloan_amt14
	       ,SUM(CASE WHEN DATEDIFF(DATE(loans.loan_time),DATE(first_loan.loan_time)) BETWEEN 0 AND 30 * 15 THEN loans.loan_amt END) reloan_amt15
	       ,SUM(CASE WHEN DATEDIFF(DATE(loans.loan_time),DATE(first_loan.loan_time)) BETWEEN 0 AND 30 * 16 THEN loans.loan_amt END) reloan_amt16
	       ,SUM(CASE WHEN DATEDIFF(DATE(loans.loan_time),DATE(first_loan.loan_time)) BETWEEN 0 AND 30 * 17 THEN loans.loan_amt END) reloan_amt17
	       ,SUM(CASE WHEN DATEDIFF(DATE(loans.loan_time),DATE(first_loan.loan_time)) BETWEEN 0 AND 30 * 18 THEN loans.loan_amt END) reloan_amt18
	FROM
	(
		SELECT  *
		FROM order_xujia_info
		WHERE 授信金额区间 IN ('[0-1000]', '(1000-2000]') 
		AND is_虚假给额 IN ('APP虚假给额', '半流程虚假给额', 'API虚假给额') 
	) first_loan
	LEFT JOIN
	(
		SELECT  *
		FROM xyf_dws.dws_inloan_user_order_df
		WHERE pt = MAX_PT('xyf_dws.dws_inloan_user_order_df')
		AND business_line IN ('APP', '小程序端')
		AND app IN ('xyf01', 'fxk')
		AND loan_status = 'success'
		AND loan_flag <> '首贷'
		AND DATE(loan_time) >= '2021-01-01' 
	) loans --cust_no level 看首贷以后的复贷 
	ON first_loan.cust_no = loans.cust_no
	GROUP BY  first_loan.user_no
	         ,first_loan.cust_no
	         ,substr(first_loan.loan_time,1,7)
	         ,DATE(first_loan.loan_time)
	         ,first_loan.loan_amt
),

In [ ]:
import query_analysis_tool as qat

data_stats = qat.run_query(query)

str_cols = [
    'mon1', '放款状态', '新老客', '资产类型', '授信金额', '授信评级',
    '是否额外放开', '飞跃会员', '飞享会员', '月收入', '学历',
    'is_虚假给额', '三个月百融查征', '风险原始定价', 'overdue1'
]

int_cols = [
    'cnt',
    '保险咨询', '催收问题', '其他', '其他退款', '减免费退款', '征信问题', '放款', '现金贷退款',
    '电销问题', '系统问题', '证明问题', '账务', '费用问题', '逾期费退款', '黑猫系统投诉',
    '保险咨询_重渠', '催收问题_重渠', '其他_重渠', '其他退款_重渠', '减免费退款_重渠', '征信问题_重渠',
    '放款_重渠', '现金贷退款_重渠', '电销问题_重渠', '系统问题_重渠', '证明问题_重渠',
    '账务_重渠', '费用问题_重渠', '逾期费退款_重渠', '黑猫系统投诉_重渠',
    '投诉数量', '重渠投诉数量'
]

float_cols = []

data_stats = qat.format_dataframe_columns(
    data_stats,
    str_cols=str_cols,
    date_cols=[], 
    int_cols=int_cols,
    float_cols=float_cols
)
qat.write_dataframe_to_excel(
    file_path=r"D:\9.极限给额专题分析\新客分析\极限给额专项分析.xlsx",
    dataframes_dict={"投诉数据": data_stats},
    start_row=1,
    include_header=True
)

In [ ]:
--- 更新版
DROP TABLE IF EXISTS lss_ewfk_ltv;
CREATE TABLE lss_ewfk_ltv AS

WITH
-- 日期周期参考表 
 cycle_ref AS
(
	SELECT  a.day_id_iso                                                  AS single_day
	       ,b.day_id_iso                                                  AS cycle_end_date
	       ,DATEDIFF(to_date(b.day_id_iso),to_date(a.day_id_iso),'dd')/30 AS mob
	FROM
	(
		SELECT  day_id_iso
		       ,1 AS tmp
		FROM xyf_dim.dim_pub_date
	) a
	INNER JOIN
	(
		SELECT  day_id_iso
		       ,1 AS tmp
		FROM xyf_dim.dim_pub_date
	) b
	ON a.day_id_iso >= '2025-01-01' AND a.day_id_iso <= DATE(getdate()) AND b.day_id_iso <= DATE(getdate())
	AND a.tmp = b.tmp AND DATEDIFF(to_date(b.day_id_iso), to_date(a.day_id_iso), 'day') IN (30, 60, 90, 120, 150, 180, 210, 240, 270, 300, 330, 360, 390, 420, 450, 480, 510, 540)
	ORDER BY a.day_id_iso, b.day_id_iso
)

--============================
--复贷系数
--============================
fudai_agg_amt AS (
SELECT  first_order_number 
       ,SUM(CASE WHEN mob = 1 THEN loan_amt END)  AS mob1_累计复贷金额
       ,SUM(CASE WHEN mob = 2 THEN loan_amt END)  AS mob2_累计复贷金额
       ,SUM(CASE WHEN mob = 3 THEN loan_amt END)  AS mob3_累计复贷金额
       ,SUM(CASE WHEN mob = 4 THEN loan_amt END)  AS mob4_累计复贷金额
       ,SUM(CASE WHEN mob = 5 THEN loan_amt END)  AS mob5_累计复贷金额
       ,SUM(CASE WHEN mob = 6 THEN loan_amt END)  AS mob6_累计复贷金额
       ,SUM(CASE WHEN mob = 7 THEN loan_amt END)  AS mob7_累计复贷金额
       ,SUM(CASE WHEN mob = 8 THEN loan_amt END)  AS mob8_累计复贷金额
       ,SUM(CASE WHEN mob = 9 THEN loan_amt END)  AS mob9_累计复贷金额
       ,SUM(CASE WHEN mob = 10 THEN loan_amt END) AS mob10_累计复贷金额
       ,SUM(CASE WHEN mob = 11 THEN loan_amt END) AS mob11_累计复贷金额
       ,SUM(CASE WHEN mob = 12 THEN loan_amt END) AS mob12_累计复贷金额
       ,SUM(CASE WHEN mob = 13 THEN loan_amt END) AS mob13_累计复贷金额
       ,SUM(CASE WHEN mob = 14 THEN loan_amt END) AS mob14_累计复贷金额
       ,SUM(CASE WHEN mob = 15 THEN loan_amt END) AS mob15_累计复贷金额
       ,SUM(CASE WHEN mob = 16 THEN loan_amt END) AS mob16_累计复贷金额
       ,SUM(CASE WHEN mob = 17 THEN loan_amt END) AS mob17_累计复贷金额
       ,SUM(CASE WHEN mob = 18 THEN loan_amt END) AS mob18_累计复贷金额
FROM
(
	SELECT  order_xujia_info.first_order_number
	       ,fudai_orders.order_number
	       ,fudai_orders.loan_time
	       ,fudai_orders.loan_amt
	       ,fudai_orders.fee_rate
	       ,fudai_orders.period
	       ,cycle_ref.mob
	       ,cycle_ref.cycle_end_date
	FROM 
	( 
		SELECT * 
		FROM xyf_jingying.lss_ewfk_first_loan_info
	) order_xujia_info
	LEFT JOIN cycle_ref
	ON DATE(order_xujia_info.first_order_time) = cycle_ref.single_day 
	LEFT JOIN
	(
		SELECT  order_number
		       ,user_no
		       ,cust_no
		       ,loan_time
		       ,loan_amt
		       ,period
		       ,fee_rate
		FROM xyf_dws.dws_inloan_user_order_df
		WHERE pt = MAX_PT('xyf_dws.dws_inloan_user_order_df')
		AND loan_status = 'success'
		AND app IN ('xyf01', 'fxk')
		AND loan_flag IN ('复贷', '加贷')
		AND business_line IN ('APP', '小程序端') -- APP上的复贷订单 
	) fudai_orders
	ON order_xujia_info.cust_no = fudai_orders.cust_no AND order_xujia_info.loan_time <= fudai_orders.loan_time AND DATE(fudai_orders.loan_time) <= cycle_ref.cycle_end_date
)
GROUP BY  first_order_number 
) ,

--============================
--商业化收入
--============================
vip AS(
SELECT  app_user_id
       ,cust_no
       ,order_time
       ,pay_time        AS tran_time
       ,real_card_price AS pay_amt
       ,0               AS refund_amt
       ,'leap'          AS card_type
FROM xyf_dwd.dwd_inloan_leap_vip_order_hf
WHERE pt = MAX_PT('xyf_dwd.dwd_inloan_leap_vip_order_hf')
AND DATE(order_time) >= '2025-05-24'
AND pay_time IS NOT NULL 

UNION ALL

SELECT  app_user_id
       ,cust_no
       ,order_time
       ,act_refund_time AS tran_time
       ,0               AS pay_amt
       ,refund_amount   AS refund_amt
       ,'leap'          AS card_type
FROM xyf_dwd.dwd_inloan_leap_vip_order_hf
WHERE pt = MAX_PT('xyf_dwd.dwd_inloan_leap_vip_order_hf')
AND DATE(order_time) >= '2025-05-24'
AND act_refund_time IS NOT NULL

UNION ALL

SELECT  app_user_id
       ,cust_no
       ,order_time
       ,pay_time            AS tran_time
       ,real_card_price/100 AS pay_amt
       ,0                   AS refund_amt
       ,'vip'               AS card_type
FROM xyf_dwd.dwd_user_vip_order_df
WHERE pt = MAX_PT('xyf_dwd.dwd_user_vip_order_df')
AND vip_card_type = 1
AND if_validation <> 0
AND pay_time IS NOT NULL 

UNION ALL

SELECT  app_user_id
       ,cust_no
       ,order_time
       ,act_refund_time   AS tran_time
       ,0                 AS pay_amt
       ,refund_amount/100 AS refund_amt
       ,'vip'             AS card_type
FROM xyf_dwd.dwd_user_vip_order_df
WHERE pt = MAX_PT('xyf_dwd.dwd_user_vip_order_df')
AND vip_card_type = 1
AND if_validation <> 0
AND act_refund_time IS NOT NULL 

UNION ALL

SELECT  app_user_id
       ,cust_no
       ,order_time
       ,order_time       AS tran_time
       ,real_order_price AS pay_amt
       ,0                AS refund_amt
       ,'tek'            AS card_type
FROM xyf_dwd.dwd_user_tek_order_df
WHERE pt = MAX_PT('xyf_dwd.dwd_user_tek_order_df') 

UNION ALL

SELECT  app_user_id
       ,cust_no
       ,order_time
       ,act_refund_time AS tran_time
       ,0               AS pay_amt
       ,refund_amount   AS refund_amt
       ,'tek'           AS card_type
FROM xyf_dwd.dwd_user_tek_order_df
WHERE pt = MAX_PT('xyf_dwd.dwd_user_tek_order_df')
AND act_refund_time IS NOT NULL 
), 

vip_revenue AS
(
SELECT  cust_no
       ,SUM(CASE WHEN mob = 1 THEN pay_amt-refund_amt END)  AS mob1_vip_revenue
       ,SUM(CASE WHEN mob = 2 THEN pay_amt-refund_amt END)  AS mob2_vip_revenue
       ,SUM(CASE WHEN mob = 3 THEN pay_amt-refund_amt END)  AS mob3_vip_revenue
       ,SUM(CASE WHEN mob = 4 THEN pay_amt-refund_amt END)  AS mob4_vip_revenue
       ,SUM(CASE WHEN mob = 5 THEN pay_amt-refund_amt END)  AS mob5_vip_revenue
       ,SUM(CASE WHEN mob = 6 THEN pay_amt-refund_amt END)  AS mob6_vip_revenue
       ,SUM(CASE WHEN mob = 7 THEN pay_amt-refund_amt END)  AS mob7_vip_revenue
       ,SUM(CASE WHEN mob = 8 THEN pay_amt-refund_amt END)  AS mob8_vip_revenue
       ,SUM(CASE WHEN mob = 9 THEN pay_amt-refund_amt END)  AS mob9_vip_revenue
       ,SUM(CASE WHEN mob = 10 THEN pay_amt-refund_amt END) AS mob10_vip_revenue
       ,SUM(CASE WHEN mob = 11 THEN pay_amt-refund_amt END) AS mob11_vip_revenue
       ,SUM(CASE WHEN mob = 12 THEN pay_amt-refund_amt END) AS mob12_vip_revenue
       ,SUM(CASE WHEN mob = 13 THEN pay_amt-refund_amt END) AS mob13_vip_revenue
       ,SUM(CASE WHEN mob = 14 THEN pay_amt-refund_amt END) AS mob14_vip_revenue
       ,SUM(CASE WHEN mob = 15 THEN pay_amt-refund_amt END) AS mob15_vip_revenue
       ,SUM(CASE WHEN mob = 16 THEN pay_amt-refund_amt END) AS mob16_vip_revenue
       ,SUM(CASE WHEN mob = 17 THEN pay_amt-refund_amt END) AS mob17_vip_revenue
FROM
(
	SELECT  order_xujia_info.cust_no
	       ,cycle_ref.mob
	       ,SUM(nvl(vip.pay_amt,0))    AS pay_amt
	       ,SUM(nvl(vip.refund_amt,0)) AS refund_amt
	FROM 
	( 
		SELECT * 
		FROM xyf_jingying.lss_ewfk_first_loan_info
	)  order_xujia_info
	LEFT JOIN cycle_ref
	ON DATE(order_xujia_info.first_order_time) = cycle_ref.single_day
	LEFT JOIN vip
	ON order_xujia_info.cust_no = vip.cust_no AND DATE(vip.tran_time) <= cycle_ref.cycle_end_date
	GROUP BY  order_xujia_info.cust_no
	         ,cycle_ref.mob
)
GROUP BY  cust_no
)


SELECT  order_with_score.*
       ,CASE WHEN order_with_score.扣得率打分 BETWEEN 0 AND 0.11289 THEN 0.910
             WHEN order_with_score.扣得率打分 BETWEEN 0.11289 AND 0.15216 THEN 0.872
             WHEN order_with_score.扣得率打分 BETWEEN 0.15216 AND 0.18588 THEN 0.839
             WHEN order_with_score.扣得率打分 BETWEEN 0.18588 AND 0.21916 THEN 0.803
             WHEN order_with_score.扣得率打分 BETWEEN 0.21916 AND 0.25826 THEN 0.798
             WHEN order_with_score.扣得率打分 BETWEEN 0.25826 AND 0.30098 THEN 0.737
             WHEN order_with_score.扣得率打分 BETWEEN 0.30098 AND 0.34784 THEN 0.706
             WHEN order_with_score.扣得率打分 BETWEEN 0.34784 AND 0.39892 THEN 0.687
             WHEN order_with_score.扣得率打分 BETWEEN 0.39892 AND 0.46428 THEN 0.606
             WHEN order_with_score.扣得率打分 BETWEEN 0.46428 AND 1 THEN 0.557 END AS 会员卡扣得率系数
       ,risk.*
	   ,fudai_agg_amt.*EXCEPT(first_order_number)
	   ,vip_revenue.*EXCEPT(cust_no)
FROM
(
	SELECT  order_xujia_info.*
	       ,COALESCE(vip_score_recall.vip_model_score,vip_score_all.vip_model_score)  AS 扣得率打分
	       ,COALESCE(vip_score_recall.decision_time,vip_score_all.decision_time)      AS 打分时间
	FROM order_xujia_info 
	-- 回溯&线上数据表, 全量用户都调用计算vip_card_sd_getfee_v2_no_final_amt_1001_prob，在调用人行数据后根据该系数计算最终扣得率系数，case_when里是最终扣得率系数的映射关系
	LEFT JOIN
	(
		SELECT  main_order_number
		       ,vip_card_sd_getfee_v2_no_final_amt_1001_prob AS vip_model_score
			   ,decision_time
		FROM xyf_jingying_dev.vip_card_sd_getfee_v2_recall_20260206
	) vip_score_recall
	ON order_xujia_info.first_order_number = vip_score_recall.main_order_number
	-- xyf_fengkong_dev.mod_recall_v2_vip_card_sd_getfee_v2_20251204112515_20260206
	-- 用1204上线的扣得率模型回溯的25年1月1日之后的首贷订单数据
	LEFT JOIN
	(
		SELECT  biz_flow_number
		       ,vip_card_sd_getfee_v2_no_final_amt_1001_prob AS vip_model_score
		       ,decision_time
		FROM xyf_dwd.dwd_risk_model_vip_card_sd_getfee_v2_di
		WHERE pt >= '20251204' -- 每个pt仅存储当天的打分数据，故pt取大于等于起始日期，pt最小为20251205，模型上线时间
		QUALIFY ROW_NUMBER() OVER (PARTITION BY biz_flow_number ORDER BY decision_time DESC) = 1 --取每个授信流水号最新的一次打分
	) vip_score_all
	ON order_xujia_info.biz_flow_number = vip_score_all.biz_flow_number
) order_with_score
-- 关联风险逾期数据
LEFT JOIN
(
	SELECT  ori_order_number
	       ,CASE WHEN y0_1_1 = 1 THEN 1 END              AS 出账1_1订单
	       ,CASE WHEN y0_1_1 = 1 THEN due_amt/100 END    AS 出账1_1应还本金
	       ,CASE WHEN y0_1_1 = 1 THEN y3_1_1/100 END     AS 出账1_1剩余应还本金
	       ,CASE WHEN y0_1_2 = 1 THEN 1 END              AS 出账1_2订单
	       ,CASE WHEN y0_1_2 = 1 THEN due_amt/100 END    AS 出账1_2应还本金
	       ,CASE WHEN y0_1_2 = 1 THEN y3_1_2/100 END     AS 出账1_2剩余应还本金
	       ,CASE WHEN y0_1_3 = 1 THEN 1 END              AS 出账1_3订单
	       ,CASE WHEN y0_1_3 = 1 THEN due_amt/100 END    AS 出账1_3应还本金
	       ,CASE WHEN y0_1_3 = 1 THEN y3_1_3/100 END     AS 出账1_3剩余应还本金
	       ,CASE WHEN y0_1_4 = 1 THEN 1 END              AS 出账1_4订单
	       ,CASE WHEN y0_1_4 = 1 THEN due_amt/100 END    AS 出账1_4应还本金
	       ,CASE WHEN y0_1_4 = 1 THEN y3_1_3/100 END     AS 出账1_4剩余应还本金
	       ,CASE WHEN y0_1_7 = 1 THEN 1 END              AS 出账1_7订单
	       ,CASE WHEN y0_1_7 = 1 THEN due_amt/100 END    AS 出账1_7应还本金
	       ,CASE WHEN y0_1_7 = 1 THEN y3_1_7/100 END     AS 出账1_7剩余应还本金
	       ,CASE WHEN y0_1_15 = 1 THEN 1 END             AS 出账1_15订单
	       ,CASE WHEN y0_1_15 = 1 THEN due_amt/100 END   AS 出账1_15应还本金
	       ,CASE WHEN y0_1_15 = 1 THEN y3_1_15/100 END   AS 出账1_15剩余应还本金
	       ,CASE WHEN y0_1_30 = 1 THEN 1 END             AS 出账1_30订单
	       ,CASE WHEN y0_1_30 = 1 THEN due_amt/100 END   AS 出账1_30应还本金
	       ,CASE WHEN y0_1_30 = 1 THEN y3_1_30/100 END   AS 出账1_30剩余应还本金
	       ,CASE WHEN y0_2_7 = 1 THEN 1 END              AS 出账2_7订单
	       ,CASE WHEN y0_2_7 = 1 THEN due_amt/100 END    AS 出账2_7应还本金
	       ,CASE WHEN y0_2_7 = 1 THEN y3_2_7/100 END     AS 出账2_7剩余应还本金
	       ,CASE WHEN y0_2_15 = 1 THEN 1 END             AS 出账2_15订单
	       ,CASE WHEN y0_2_15 = 1 THEN due_amt/100 END   AS 出账2_15应还本金
	       ,CASE WHEN y0_2_15 = 1 THEN y3_2_15/100 END   AS 出账2_15剩余应还本金
	       ,CASE WHEN y0_2_30 = 1 THEN 1 END             AS 出账2_30订单
	       ,CASE WHEN y0_2_30 = 1 THEN due_amt/100 END   AS 出账2_30应还本金
	       ,CASE WHEN y0_2_30 = 1 THEN y3_2_30/100 END   AS 出账2_30剩余应还本金
	       ,CASE WHEN y0_3_7 = 1 THEN 1 END              AS 出账3_7订单
	       ,CASE WHEN y0_3_7 = 1 THEN due_amt/100 END    AS 出账3_7应还本金
	       ,CASE WHEN y0_3_7 = 1 THEN y3_3_7/100 END     AS 出账3_7剩余应还本金
	       ,CASE WHEN y0_3_15 = 1 THEN 1 END             AS 出账3_15订单
	       ,CASE WHEN y0_3_15 = 1 THEN due_amt/100 END   AS 出账3_15应还本金
	       ,CASE WHEN y0_3_15 = 1 THEN y3_3_15/100 END   AS 出账3_15剩余应还本金
	       ,CASE WHEN y0_3_30 = 1 THEN 1 END             AS 出账3_30订单
	       ,CASE WHEN y0_3_30 = 1 THEN due_amt/100 END   AS 出账3_30应还本金
	       ,CASE WHEN y0_3_30 = 1 THEN y3_3_30/100 END   AS 出账3_30剩余应还本金
	       ,CASE WHEN y0_4_30 = 1 THEN 1 END             AS 出账4_30订单
	       ,CASE WHEN y0_4_30 = 1 THEN due_amt/100 END   AS 出账4_30应还本金
	       ,CASE WHEN y0_4_30 = 1 THEN y3_4_30/100 END   AS 出账4_30剩余应还本金
	       ,CASE WHEN y0_5_30 = 1 THEN 1 END             AS 出账5_30订单
	       ,CASE WHEN y0_5_30 = 1 THEN due_amt/100 END   AS 出账5_30应还本金
	       ,CASE WHEN y0_5_30 = 1 THEN y3_5_30/100 END   AS 出账5_30剩余应还本金
	       ,CASE WHEN y0_6_30 = 1 THEN 1 END             AS 出账6_30订单
	       ,CASE WHEN y0_6_30 = 1 THEN due_amt/100 END   AS 出账6_30应还本金
	       ,CASE WHEN y0_6_30 = 1 THEN y3_6_30/100 END   AS 出账6_30剩余应还本金
	       ,CASE WHEN y0_7_30 = 1 THEN 1 END             AS 出账7_30订单
	       ,CASE WHEN y0_7_30 = 1 THEN due_amt/100 END   AS 出账7_30应还本金
	       ,CASE WHEN y0_7_30 = 1 THEN y3_7_30/100 END   AS 出账7_30剩余应还本金
	       ,CASE WHEN y0_8_30 = 1 THEN 1 END             AS 出账8_30订单
	       ,CASE WHEN y0_8_30 = 1 THEN due_amt/100 END   AS 出账8_30应还本金
	       ,CASE WHEN y0_8_30 = 1 THEN y3_8_30/100 END   AS 出账8_30剩余应还本金
	       ,CASE WHEN y0_9_30 = 1 THEN 1 END             AS 出账9_30订单
	       ,CASE WHEN y0_9_30 = 1 THEN due_amt/100 END   AS 出账9_30应还本金
	       ,CASE WHEN y0_9_30 = 1 THEN y3_9_30/100 END   AS 出账9_30剩余应还本金
	       ,CASE WHEN y0_10_30 = 1 THEN 1 END            AS 出账10_30订单
	       ,CASE WHEN y0_10_30 = 1 THEN due_amt/100 END  AS 出账10_30应还本金
	       ,CASE WHEN y0_10_30 = 1 THEN y3_10_30/100 END AS 出账10_30剩余应还本金
	       ,CASE WHEN y0_11_30 = 1 THEN 1 END            AS 出账11_30订单
	       ,CASE WHEN y0_11_30 = 1 THEN due_amt/100 END  AS 出账11_30应还本金
	       ,CASE WHEN y0_11_30 = 1 THEN y3_11_30/100 END AS 出账11_30剩余应还本金
	       ,CASE WHEN y0_12_30 = 1 THEN 1 END            AS 出账12_30订单
	       ,CASE WHEN y0_12_30 = 1 THEN due_amt/100 END  AS 出账12_30应还本金
	       ,CASE WHEN y0_12_30 = 1 THEN y3_12_30/100 END AS 出账12_30剩余应还本金
	       ,CASE WHEN y0_13_30 = 1 THEN 1 END            AS 出账13_30订单
	       ,CASE WHEN y0_13_30 = 1 THEN due_amt/100 END  AS 出账13_30应还本金
	       ,CASE WHEN y0_13_30 = 1 THEN y3_13_30/100 END AS 出账13_30剩余应还本金
	       ,CASE WHEN y0_14_30 = 1 THEN 1 END            AS 出账14_30订单
	       ,CASE WHEN y0_14_30 = 1 THEN due_amt/100 END  AS 出账14_30应还本金
	       ,CASE WHEN y0_14_30 = 1 THEN y3_14_30/100 END AS 出账14_30剩余应还本金
	       ,CASE WHEN y0_15_30 = 1 THEN 1 END            AS 出账15_30订单
	       ,CASE WHEN y0_15_30 = 1 THEN due_amt/100 END  AS 出账15_30应还本金
	       ,CASE WHEN y0_15_30 = 1 THEN y3_15_30/100 END AS 出账15_30剩余应还本金
	FROM xyf_dws.dws_repay_risk_order_bill_mob_agg_df ---这个是大额拆单后的新表, 区别在于，用户1笔拆成多笔之后有1笔违约就算全部违约。 
	WHERE pt = '${bizdate}' 
) risk
ON order_with_score.first_order_number = risk.ori_order_number

LEFT JOIN  fudai_agg_amt
ON order_with_score.first_order_number = fudai_agg_amt.first_order_number

LEFT JOIN vip_revenue
ON order_with_score.cust_no = vip_revenue.cust_no


####  复贷逾期

In [9]:
query = '''
-- 复贷逾期与期限
WITH lss_ewfk_fudai_info AS
(
	SELECT  fudai_orders_info.*
	       ,risk.*EXCEPT(ori_order_number)
	FROM
	(
		SELECT  lss_ewfk_ltv.授信金额区间
		       ,lss_ewfk_ltv.is_虚假给额
		       ,fudai_orders.*
		FROM lss_ewfk_ltv
		INNER JOIN
		(
			SELECT  DATE(first_order_time)       AS 订单发起日期
			       ,SUBSTR(first_order_time,1,7) AS 订单发起月
			       ,DATE(loan_time)              AS 放款日期
			       ,SUBSTR(loan_time,1,7)        AS 放款月
			       ,order_number
			       ,user_no
			       ,cust_no
			       ,first_order_number
			       ,app
			       ,inner_app
			       ,business_line
			       ,loan_flag
			       ,first_order_time
			       ,loan_time
			       ,loan_amt
			       ,period
			       ,asset_type_flag
			       ,fee_rate
			       ,biz_flow_number
			FROM xyf_dws.dws_inloan_user_order_df
			WHERE pt = MAX_PT('xyf_dws.dws_inloan_user_order_df')
			AND loan_status = 'success'
			AND app IN ('xyf01', 'fxk')
			AND loan_flag IN ('复贷', '加贷')
			AND business_line IN ('APP', '小程序端') -- APP上的复贷订单 
 
		) fudai_orders
		ON lss_ewfk_ltv.cust_no = fudai_orders.cust_no AND lss_ewfk_ltv.loan_time <= fudai_orders.loan_time
	) fudai_orders_info
	LEFT JOIN
	(
		SELECT  ori_order_number
		       ,CASE WHEN y0_1_1 = 1 THEN 1 END              AS 出账1_1订单
		       ,CASE WHEN y0_1_1 = 1 THEN due_amt/100 END    AS 出账1_1应还本金
		       ,CASE WHEN y0_1_1 = 1 THEN y3_1_1/100 END     AS 出账1_1剩余应还本金
		       ,CASE WHEN y0_1_2 = 1 THEN 1 END              AS 出账1_2订单
		       ,CASE WHEN y0_1_2 = 1 THEN due_amt/100 END    AS 出账1_2应还本金
		       ,CASE WHEN y0_1_2 = 1 THEN y3_1_2/100 END     AS 出账1_2剩余应还本金
		       ,CASE WHEN y0_1_3 = 1 THEN 1 END              AS 出账1_3订单
		       ,CASE WHEN y0_1_3 = 1 THEN due_amt/100 END    AS 出账1_3应还本金
		       ,CASE WHEN y0_1_3 = 1 THEN y3_1_3/100 END     AS 出账1_3剩余应还本金
		       ,CASE WHEN y0_1_4 = 1 THEN 1 END              AS 出账1_4订单
		       ,CASE WHEN y0_1_4 = 1 THEN due_amt/100 END    AS 出账1_4应还本金
		       ,CASE WHEN y0_1_4 = 1 THEN y3_1_3/100 END     AS 出账1_4剩余应还本金
		       ,CASE WHEN y0_1_7 = 1 THEN 1 END              AS 出账1_7订单
		       ,CASE WHEN y0_1_7 = 1 THEN due_amt/100 END    AS 出账1_7应还本金
		       ,CASE WHEN y0_1_7 = 1 THEN y3_1_7/100 END     AS 出账1_7剩余应还本金
		       ,CASE WHEN y0_1_15 = 1 THEN 1 END             AS 出账1_15订单
		       ,CASE WHEN y0_1_15 = 1 THEN due_amt/100 END   AS 出账1_15应还本金
		       ,CASE WHEN y0_1_15 = 1 THEN y3_1_15/100 END   AS 出账1_15剩余应还本金
		       ,CASE WHEN y0_1_30 = 1 THEN 1 END             AS 出账1_30订单
		       ,CASE WHEN y0_1_30 = 1 THEN due_amt/100 END   AS 出账1_30应还本金
		       ,CASE WHEN y0_1_30 = 1 THEN y3_1_30/100 END   AS 出账1_30剩余应还本金
		       ,CASE WHEN y0_2_7 = 1 THEN 1 END              AS 出账2_7订单
		       ,CASE WHEN y0_2_7 = 1 THEN due_amt/100 END    AS 出账2_7应还本金
		       ,CASE WHEN y0_2_7 = 1 THEN y3_2_7/100 END     AS 出账2_7剩余应还本金
		       ,CASE WHEN y0_2_15 = 1 THEN 1 END             AS 出账2_15订单
		       ,CASE WHEN y0_2_15 = 1 THEN due_amt/100 END   AS 出账2_15应还本金
		       ,CASE WHEN y0_2_15 = 1 THEN y3_2_15/100 END   AS 出账2_15剩余应还本金
		       ,CASE WHEN y0_2_30 = 1 THEN 1 END             AS 出账2_30订单
		       ,CASE WHEN y0_2_30 = 1 THEN due_amt/100 END   AS 出账2_30应还本金
		       ,CASE WHEN y0_2_30 = 1 THEN y3_2_30/100 END   AS 出账2_30剩余应还本金
		       ,CASE WHEN y0_3_7 = 1 THEN 1 END              AS 出账3_7订单
		       ,CASE WHEN y0_3_7 = 1 THEN due_amt/100 END    AS 出账3_7应还本金
		       ,CASE WHEN y0_3_7 = 1 THEN y3_3_7/100 END     AS 出账3_7剩余应还本金
		       ,CASE WHEN y0_3_15 = 1 THEN 1 END             AS 出账3_15订单
		       ,CASE WHEN y0_3_15 = 1 THEN due_amt/100 END   AS 出账3_15应还本金
		       ,CASE WHEN y0_3_15 = 1 THEN y3_3_15/100 END   AS 出账3_15剩余应还本金
		       ,CASE WHEN y0_3_30 = 1 THEN 1 END             AS 出账3_30订单
		       ,CASE WHEN y0_3_30 = 1 THEN due_amt/100 END   AS 出账3_30应还本金
		       ,CASE WHEN y0_3_30 = 1 THEN y3_3_30/100 END   AS 出账3_30剩余应还本金
		       ,CASE WHEN y0_4_30 = 1 THEN 1 END             AS 出账4_30订单
		       ,CASE WHEN y0_4_30 = 1 THEN due_amt/100 END   AS 出账4_30应还本金
		       ,CASE WHEN y0_4_30 = 1 THEN y3_4_30/100 END   AS 出账4_30剩余应还本金
		       ,CASE WHEN y0_5_30 = 1 THEN 1 END             AS 出账5_30订单
		       ,CASE WHEN y0_5_30 = 1 THEN due_amt/100 END   AS 出账5_30应还本金
		       ,CASE WHEN y0_5_30 = 1 THEN y3_5_30/100 END   AS 出账5_30剩余应还本金
		       ,CASE WHEN y0_6_30 = 1 THEN 1 END             AS 出账6_30订单
		       ,CASE WHEN y0_6_30 = 1 THEN due_amt/100 END   AS 出账6_30应还本金
		       ,CASE WHEN y0_6_30 = 1 THEN y3_6_30/100 END   AS 出账6_30剩余应还本金
		       ,CASE WHEN y0_7_30 = 1 THEN 1 END             AS 出账7_30订单
		       ,CASE WHEN y0_7_30 = 1 THEN due_amt/100 END   AS 出账7_30应还本金
		       ,CASE WHEN y0_7_30 = 1 THEN y3_7_30/100 END   AS 出账7_30剩余应还本金
		       ,CASE WHEN y0_8_30 = 1 THEN 1 END             AS 出账8_30订单
		       ,CASE WHEN y0_8_30 = 1 THEN due_amt/100 END   AS 出账8_30应还本金
		       ,CASE WHEN y0_8_30 = 1 THEN y3_8_30/100 END   AS 出账8_30剩余应还本金
		       ,CASE WHEN y0_9_30 = 1 THEN 1 END             AS 出账9_30订单
		       ,CASE WHEN y0_9_30 = 1 THEN due_amt/100 END   AS 出账9_30应还本金
		       ,CASE WHEN y0_9_30 = 1 THEN y3_9_30/100 END   AS 出账9_30剩余应还本金
		       ,CASE WHEN y0_10_30 = 1 THEN 1 END            AS 出账10_30订单
		       ,CASE WHEN y0_10_30 = 1 THEN due_amt/100 END  AS 出账10_30应还本金
		       ,CASE WHEN y0_10_30 = 1 THEN y3_10_30/100 END AS 出账10_30剩余应还本金
		       ,CASE WHEN y0_11_30 = 1 THEN 1 END            AS 出账11_30订单
		       ,CASE WHEN y0_11_30 = 1 THEN due_amt/100 END  AS 出账11_30应还本金
		       ,CASE WHEN y0_11_30 = 1 THEN y3_11_30/100 END AS 出账11_30剩余应还本金
		       ,CASE WHEN y0_12_30 = 1 THEN 1 END            AS 出账12_30订单
		       ,CASE WHEN y0_12_30 = 1 THEN due_amt/100 END  AS 出账12_30应还本金
		       ,CASE WHEN y0_12_30 = 1 THEN y3_12_30/100 END AS 出账12_30剩余应还本金
		       ,CASE WHEN y0_13_30 = 1 THEN 1 END            AS 出账13_30订单
		       ,CASE WHEN y0_13_30 = 1 THEN due_amt/100 END  AS 出账13_30应还本金
		       ,CASE WHEN y0_13_30 = 1 THEN y3_13_30/100 END AS 出账13_30剩余应还本金
		       ,CASE WHEN y0_14_30 = 1 THEN 1 END            AS 出账14_30订单
		       ,CASE WHEN y0_14_30 = 1 THEN due_amt/100 END  AS 出账14_30应还本金
		       ,CASE WHEN y0_14_30 = 1 THEN y3_14_30/100 END AS 出账14_30剩余应还本金
		       ,CASE WHEN y0_15_30 = 1 THEN 1 END            AS 出账15_30订单
		       ,CASE WHEN y0_15_30 = 1 THEN due_amt/100 END  AS 出账15_30应还本金
		       ,CASE WHEN y0_15_30 = 1 THEN y3_15_30/100 END AS 出账15_30剩余应还本金
		FROM xyf_dws.dws_repay_risk_order_bill_mob_agg_df ---这个是大额拆单后的新表, 区别在于，用户1笔拆成多笔之后有1笔违约就算全部违约。 
		WHERE pt = MAX_PT('xyf_dws.dws_repay_risk_order_bill_mob_agg_df')
	) risk
	ON fudai_orders_info.first_order_number = risk.ori_order_number
)
SELECT  放款月
       ,is_虚假给额
       ,授信金额区间
       ,COUNT(DISTINCT user_no)            AS 放款人数
       ,COUNT(DISTINCT first_order_number) AS 放款订单数
       ,SUM(loan_amt)                      AS 放款金额
       ,SUM(loan_amt * fee_rate)           AS `金额_定价`
       ,SUM(loan_amt * period)             AS `金额_期限`
       ,SUM(出账1_30应还本金)                    AS 出账1_30应还本金
       ,SUM(出账2_30应还本金)                    AS 出账2_30应还本金
       ,SUM(出账3_30应还本金)                    AS 出账3_30应还本金
       ,SUM(出账4_30应还本金)                    AS 出账4_30应还本金
       ,SUM(出账5_30应还本金)                    AS 出账5_30应还本金
       ,SUM(出账6_30应还本金)                    AS 出账6_30应还本金
       ,SUM(出账7_30应还本金)                    AS 出账7_30应还本金
       ,SUM(出账8_30应还本金)                    AS 出账8_30应还本金
       ,SUM(出账9_30应还本金)                    AS 出账9_30应还本金
       ,SUM(出账10_30应还本金)                   AS 出账10_30应还本金
       ,SUM(出账11_30应还本金)                   AS 出账11_30应还本金
       ,SUM(出账12_30应还本金)                   AS 出账12_30应还本金
       ,SUM(出账1_1剩余应还本金)                   AS 出账1_1剩余应还本金
       ,SUM(出账1_30剩余应还本金)                  AS 出账1_30剩余应还本金
       ,SUM(出账2_30剩余应还本金)                  AS 出账2_30剩余应还本金
       ,SUM(出账3_30剩余应还本金)                  AS 出账3_30剩余应还本金
       ,SUM(出账4_30剩余应还本金)                  AS 出账4_30剩余应还本金
       ,SUM(出账5_30剩余应还本金)                  AS 出账5_30剩余应还本金
       ,SUM(出账6_30剩余应还本金)                  AS 出账6_30剩余应还本金
       ,SUM(出账7_30剩余应还本金)                  AS 出账7_30剩余应还本金
       ,SUM(出账8_30剩余应还本金)                  AS 出账8_30剩余应还本金
       ,SUM(出账9_30剩余应还本金)                  AS 出账9_30剩余应还本金
       ,SUM(出账10_30剩余应还本金)                 AS 出账10_30剩余应还本金
       ,SUM(出账11_30剩余应还本金)                 AS 出账11_30剩余应还本金
       ,SUM(出账12_30剩余应还本金)                 AS 出账12_30剩余应还本金
FROM lss_ewfk_fudai_info
WHERE 授信金额区间 IN ('[0-1000]', '(1000-2000]')
GROUP BY  放款月
         ,is_虚假给额
         ,授信金额区间
'''

#底表数据
result = o.execute_sql(query)
loan_stats = result.open_reader().to_pandas()


,放款月,is_虚假给额,授信金额区间,放款人数,放款订单数,放款金额,金额_定价,金额_期限,出账1_30应还本金,出账2_30应还本金,...,出账3_30剩余应还本金,出账4_30剩余应还本金,出账5_30剩余应还本金,出账6_30剩余应还本金,出账7_30剩余应还本金,出账8_30剩余应还本金,出账9_30剩余应还本金,出账10_30剩余应还本金,出账11_30剩余应还本金,出账12_30剩余应还本金
0,2025-01,APP虚假给额,[0-1000],147,204,235400,84720.46,2070700,234389.27,234389.27,...,24172.99,29809.93,34287.10,35548.50,35466.14,36569.72,36993.73,37258.23,37174.06,33403.28
1,2025-01,非虚假给额,(1000-2000],998,1303,1519600,543372.92,12968700,1519410.80,1519410.80,...,55709.71,72467.52,81611.25,96514.22,102834.62,105318.26,106802.16,106952.71,107548.65,95089.98
2,2025-02,非虚假给额,(1000-2000],4487,6030,7368200,2643479.38,60119900,7367478.01,7367478.01,...,389987.37,492718.70,548868.82,627752.54,657298.18,667509.32,686474.62,689312.56,675383.48,NaN
3,2025-03,非虚假给额,(1000-2000],9334,13201,15240200,5461278.38,123851100,15238120.69,15238120.69,...,693283.06,890456.20,1044504.69,1176718.45,1248716.45,1274593.80,1303702.30,1263096.70,NaN,NaN
4,2025-04,非虚假给额,[0-1000],34790,50299,52566200,18575990.98,461593200,52557795.44,52557795.44,...,3019431.10,3866007.76,4549784.80,5104655.22,5405439.17,5620062.48,5713914.21,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
56,2026-01,半流程虚假给额,[0-1000],7,8,10600,3814.94,66200,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
57,2026-03,APP虚假给额,(1000-2000],94,97,134300,47710.63,894100,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
58,2026-03,APP虚假给额,[0-1000],312,357,383900,136591.68,2332800,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
59,2026-03,半流程虚假给额,[0-1000],10,12,12600,4534.74,72600,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
# 字符串字段
str_cols = [
    '订单发起月',
	'is_虚假给额',
    '授信金额区间',
]
for col in str_cols:
    if col in loan_stats.columns:
        # loan_stats[col] = loan_stats[col].astype('string').fillna('')  # 填充空值为空字符串
        loan_stats[col] = loan_stats[col].astype('string')


# 浮点数字段 - 保留2位小数，所有有应还本金的字段均为浮点数
float_cols = [c for c in loan_stats.columns if ('应还本金' in c)] + ['放款金额', '金额_定价', '金额_期限']
float_cols = [c for c in dict.fromkeys(float_cols) if c in loan_stats.columns]  # 去重 + 存在性过滤
float_cols += [
]
for col in float_cols:
    if col in loan_stats.columns:
        loan_stats[col] = pd.to_numeric(loan_stats[col], errors='coerce').astype('float64') # “清洗 + 转类型”，对混有字符串数字（如 "12.3"）、空字符串、非法字符等也能处理
        # loan_stats[col] = loan_stats[col].astype('float64')  # 不做解析，只做 dtype 强转，遇到 "1,234"、"—"、""、'abc' 这类会直接报 ValueError
        # loan_stats[col] = pd.to_numeric(loan_stats[col], errors='coerce').fillna(0).astype('float64') # 填充空值为零，但dataworks实际空值导出也是空白，非零值

# 整数字段
int_cols = [
    '放款人数','放款订单数'
]
for col in int_cols:
    if col in loan_stats.columns:
        # loan_stats[col] = pd.to_numeric(loan_stats[col], errors='coerce').fillna(0).astype('Int64') 
        loan_stats[col] = pd.to_numeric(loan_stats[col], errors='coerce').astype('Int64')  # 使用Pandas 的 Int64（可空整数）/ NumPy 的 int64（普通整数）

file_path = r"D:\9.极限给额专题分析\极限给额LTV.xlsx"
write_dataframe_to_excel_com(file_path, {"复贷逾期数据": loan_stats}, start_row=1, include_header=True)

成功写入工作表: 复贷逾期数据
文件已保存: D:\9.极限给额专题分析\极限给额LTV.xlsx


In [11]:
calc_fields = {
    "金额加权定价": {"formula": "='金额_定价'/放款金额", "number_format": "0.00%"},
    "金额加权期限": {"formula": "='金额_期限'/放款金额", "number_format": "0.00"},
    "件均定价": {"formula": "='放款金额'/放款订单数", "number_format": "0.00"},
    "M1": {"formula": "='出账1_30剩余应还本金'/出账1_30应还本金", "number_format": "0.00%"},
    "M2": {"formula": "='出账2_30剩余应还本金'/出账2_30应还本金", "number_format": "0.00%"},
    "M3": {"formula": "='出账3_30剩余应还本金'/出账3_30应还本金", "number_format": "0.00%"},
    "M4": {"formula": "='出账4_30剩余应还本金'/出账4_30应还本金", "number_format": "0.00%"},
    "M5": {"formula": "='出账5_30剩余应还本金'/出账5_30应还本金", "number_format": "0.00%"},
    "M6": {"formula": "='出账6_30剩余应还本金'/出账6_30应还本金", "number_format": "0.00%"},
    "M7": {"formula": "='出账7_30剩余应还本金'/出账7_30应还本金", "number_format": "0.00%"},
    "M8": {"formula": "='出账8_30剩余应还本金'/出账8_30应还本金", "number_format": "0.00%"},
    "M9": {"formula": "='出账9_30剩余应还本金'/出账9_30应还本金", "number_format": "0.00%"},
    "M10": {"formula": "='出账10_30剩余应还本金'/出账10_30应还本金", "number_format": "0.00%"},
    "M11": {"formula": "='出账11_30剩余应还本金'/出账11_30应还本金", "number_format": "0.00%"},
    "M12": {"formula": "='出账12_30剩余应还本金'/出账12_30应还本金", "number_format": "0.00%"},
}

failed = add_pivot_calculated_fields_com(
    file_path=file_path,
    sheet_name="LTV分析",
    pivot_name="数据透视表3",
    fields=calc_fields,
    refresh=True,
    add_to_values=True,      # 自动加入值区域
    skip_if_exists=True,     # 同名就记录并跳过（避免第二次 Add “发生意外”）
    verbose=True,
) 

[Pivot] sheet=LTV分析 pivot=数据透视表3 cache_index=1
[Fail] Add CalculatedField 失败: 金额加权定价 | (-2147352567, '发生意外。', (0, None, None, None, 0, -2146827284), None)
[Fail] Add CalculatedField 失败: 金额加权期限 | (-2147352567, '发生意外。', (0, None, None, None, 0, -2146827284), None)
[Fail] Add CalculatedField 失败: 件均定价 | (-2147352567, '发生意外。', (0, None, None, None, 0, -2146827284), None)
[Fail] Add CalculatedField 失败: M1 | (-2147352567, '发生意外。', (0, None, None, None, 0, -2146827284), None)
[OK] Add CalculatedField: M2 | ='出账2_30剩余应还本金'/出账2_30应还本金
[OK] Add to Values: M2 | format=0.00%
[Fail] Add CalculatedField 失败: M3 | (-2147352567, '发生意外。', (0, None, None, None, 0, -2146827284), None)
[OK] Add CalculatedField: M4 | ='出账4_30剩余应还本金'/出账4_30应还本金
[OK] Add to Values: M4 | format=0.00%
[OK] Add CalculatedField: M5 | ='出账5_30剩余应还本金'/出账5_30应还本金
[OK] Add to Values: M5 | format=0.00%
[Fail] Add CalculatedField 失败: M6 | (-2147352567, '发生意外。', (0, None, None, None, 0, -2146827284), None)
[OK] Add CalculatedField: M7 | 

In [ ]:
## SQL查询
query='''
SELECT  放款月
       ,asset_type_flag
       ,period
       ,授信渠道
       ,is_虚假给额
       ,会员卡扣得率系数
       ,授信金额区间
       ,COUNT(DISTINCT user_no)            AS 放款人数
       ,COUNT(DISTINCT first_order_number) AS 放款订单数
       ,SUM(loan_amt)                      AS 放款金额
       ,SUM(loan_amt * fee_rate) * 100     AS `金额_定价`
       ,SUM(loan_amt * period)             AS `金额_期限`
       ,SUM(出账1_1订单)                       AS 出账1_1订单
       ,SUM(出账1_2订单)                       AS 出账1_2订单
       ,SUM(出账1_3订单)                       AS 出账1_3订单
       ,SUM(出账1_4订单)                       AS 出账1_4订单
       ,SUM(出账1_7订单)                       AS 出账1_7订单
       ,SUM(出账1_15订单)                      AS 出账1_15订单
       ,SUM(出账1_30订单)                      AS 出账1_30订单
       ,SUM(出账2_7订单)                       AS 出账2_7订单
       ,SUM(出账2_15订单)                      AS 出账2_15订单
       ,SUM(出账2_30订单)                      AS 出账2_30订单
       ,SUM(出账3_7订单)                       AS 出账3_7订单
       ,SUM(出账3_15订单)                      AS 出账3_15订单
       ,SUM(出账3_30订单)                      AS 出账3_30订单
       ,SUM(出账4_30订单)                      AS 出账4_30订单
       ,SUM(出账5_30订单)                      AS 出账5_30订单
       ,SUM(出账6_30订单)                      AS 出账6_30订单
       ,SUM(出账7_30订单)                      AS 出账7_30订单
       ,SUM(出账8_30订单)                      AS 出账8_30订单
       ,SUM(出账9_30订单)                      AS 出账9_30订单
       ,SUM(出账10_30订单)                     AS 出账10_30订单
       ,SUM(出账11_30订单)                     AS 出账11_30订单
       ,SUM(出账12_30订单)                     AS 出账12_30订单
       ,SUM(出账13_30订单)                     AS 出账13_30订单
       ,SUM(出账14_30订单)                     AS 出账14_30订单
       ,SUM(出账15_30订单)                     AS 出账15_30订单
       ,SUM(出账1_1应还本金)                     AS 出账1_1应还本金
       ,SUM(出账1_2应还本金)                     AS 出账1_2应还本金
       ,SUM(出账1_3应还本金)                     AS 出账1_3应还本金
       ,SUM(出账1_4应还本金)                     AS 出账1_4应还本金
       ,SUM(出账1_7应还本金)                     AS 出账1_7应还本金
       ,SUM(出账1_15应还本金)                    AS 出账1_15应还本金
       ,SUM(出账1_30应还本金)                    AS 出账1_30应还本金
       ,SUM(出账2_7应还本金)                     AS 出账2_7应还本金
       ,SUM(出账2_15应还本金)                    AS 出账2_15应还本金
       ,SUM(出账2_30应还本金)                    AS 出账2_30应还本金
       ,SUM(出账3_7应还本金)                     AS 出账3_7应还本金
       ,SUM(出账3_15应还本金)                    AS 出账3_15应还本金
       ,SUM(出账3_30应还本金)                    AS 出账3_30应还本金
       ,SUM(出账4_30应还本金)                    AS 出账4_30应还本金
       ,SUM(出账5_30应还本金)                    AS 出账5_30应还本金
       ,SUM(出账6_30应还本金)                    AS 出账6_30应还本金
       ,SUM(出账7_30应还本金)                    AS 出账7_30应还本金
       ,SUM(出账8_30应还本金)                    AS 出账8_30应还本金
       ,SUM(出账9_30应还本金)                    AS 出账9_30应还本金
       ,SUM(出账10_30应还本金)                   AS 出账10_30应还本金
       ,SUM(出账11_30应还本金)                   AS 出账11_30应还本金
       ,SUM(出账12_30应还本金)                   AS 出账12_30应还本金
       ,SUM(出账13_30应还本金)                   AS 出账13_30应还本金
       ,SUM(出账14_30应还本金)                   AS 出账14_30应还本金
       ,SUM(出账15_30应还本金)                   AS 出账15_30应还本金
       ,SUM(出账1_1剩余应还本金)                   AS 出账1_1剩余应还本金
       ,SUM(出账1_2剩余应还本金)                   AS 出账1_2剩余应还本金
       ,SUM(出账1_3剩余应还本金)                   AS 出账1_3剩余应还本金
       ,SUM(出账1_4剩余应还本金)                   AS 出账1_4剩余应还本金
       ,SUM(出账1_7剩余应还本金)                   AS 出账1_7剩余应还本金
       ,SUM(出账1_15剩余应还本金)                  AS 出账1_15剩余应还本金
       ,SUM(出账1_30剩余应还本金)                  AS 出账1_30剩余应还本金
       ,SUM(出账2_7剩余应还本金)                   AS 出账2_7剩余应还本金
       ,SUM(出账2_15剩余应还本金)                  AS 出账2_15剩余应还本金
       ,SUM(出账2_30剩余应还本金)                  AS 出账2_30剩余应还本金
       ,SUM(出账3_7剩余应还本金)                   AS 出账3_7剩余应还本金
       ,SUM(出账3_15剩余应还本金)                  AS 出账3_15剩余应还本金
       ,SUM(出账3_30剩余应还本金)                  AS 出账3_30剩余应还本金
       ,SUM(出账4_30剩余应还本金)                  AS 出账4_30剩余应还本金
       ,SUM(出账5_30剩余应还本金)                  AS 出账5_30剩余应还本金
       ,SUM(出账6_30剩余应还本金)                  AS 出账6_30剩余应还本金
       ,SUM(出账7_30剩余应还本金)                  AS 出账7_30剩余应还本金
       ,SUM(出账8_30剩余应还本金)                  AS 出账8_30剩余应还本金
       ,SUM(出账9_30剩余应还本金)                  AS 出账9_30剩余应还本金
       ,SUM(出账10_30剩余应还本金)                 AS 出账10_30剩余应还本金
       ,SUM(出账11_30剩余应还本金)                 AS 出账11_30剩余应还本金
       ,SUM(出账12_30剩余应还本金)                 AS 出账12_30剩余应还本金
       ,SUM(出账13_30剩余应还本金)                 AS 出账13_30剩余应还本金
       ,SUM(出账14_30剩余应还本金)                 AS 出账14_30剩余应还本金
       ,SUM(出账15_30剩余应还本金)                 AS 出账15_30剩余应还本金
       ,SUM(mob1_累计复贷金额)                   AS mob1_累计复贷金额
       ,SUM(mob2_累计复贷金额)                   AS mob2_累计复贷金额
       ,SUM(mob3_累计复贷金额)                   AS mob3_累计复贷金额
       ,SUM(mob4_累计复贷金额)                   AS mob4_累计复贷金额
       ,SUM(mob5_累计复贷金额)                   AS mob5_累计复贷金额
       ,SUM(mob6_累计复贷金额)                   AS mob6_累计复贷金额
       ,SUM(mob7_累计复贷金额)                   AS mob7_累计复贷金额
       ,SUM(mob8_累计复贷金额)                   AS mob8_累计复贷金额
       ,SUM(mob9_累计复贷金额)                   AS mob9_累计复贷金额
       ,SUM(mob10_累计复贷金额)                  AS mob10_累计复贷金额
       ,SUM(mob11_累计复贷金额)                  AS mob11_累计复贷金额
       ,SUM(mob12_累计复贷金额)                  AS mob12_累计复贷金额
       ,SUM(mob13_累计复贷金额)                  AS mob13_累计复贷金额
       ,SUM(mob14_累计复贷金额)                  AS mob14_累计复贷金额
       ,SUM(mob15_累计复贷金额)                  AS mob15_累计复贷金额
       ,SUM(mob16_累计复贷金额)                  AS mob16_累计复贷金额
       ,SUM(mob17_累计复贷金额)                  AS mob17_累计复贷金额
       ,SUM(mob18_累计复贷金额)                  AS mob18_累计复贷金额
       ,SUM(mob1_vip_revenue)              AS mob1_vip_revenue
       ,SUM(mob2_vip_revenue)              AS mob2_vip_revenue
       ,SUM(mob3_vip_revenue)              AS mob3_vip_revenue
       ,SUM(mob4_vip_revenue)              AS mob4_vip_revenue
       ,SUM(mob5_vip_revenue)              AS mob5_vip_revenue
       ,SUM(mob6_vip_revenue)              AS mob6_vip_revenue
       ,SUM(mob7_vip_revenue)              AS mob7_vip_revenue
       ,SUM(mob8_vip_revenue)              AS mob8_vip_revenue
       ,SUM(mob9_vip_revenue)              AS mob9_vip_revenue
       ,SUM(mob10_vip_revenue)             AS mob10_vip_revenue
       ,SUM(mob11_vip_revenue)             AS mob11_vip_revenue
       ,SUM(mob12_vip_revenue)             AS mob12_vip_revenue
FROM lss_ewfk_ltv
WHERE 授信金额区间 IN ('[0-1000]', '(1000-2000]') 
-- AND is_虚假给额 IN ('APP虚假给额', '半流程虚假给额', 'API虚假给额')
GROUP BY  放款月
         ,asset_type_flag
         ,period
         ,授信渠道
         ,is_虚假给额
         ,会员卡扣得率系数
         ,授信金额区间
'''

In [ ]:
#底表数据
result = o.execute_sql(query)
loan_stats = result.open_reader().to_pandas()

# 字符串字段
str_cols = [
    '订单发起月',
    'asset_type_flag',
    'period',
    '授信渠道',
    'is_虚假给额',
    '是否买卡订单',
    '会员卡扣得率系数',
    '授信金额区间',
]
for col in str_cols:
    if col in loan_stats.columns:
        # loan_stats[col] = loan_stats[col].astype('string').fillna('')  # 填充空值为空字符串
        loan_stats[col] = loan_stats[col].astype('string')


# 浮点数字段 - 保留2位小数，所有有应还本金的字段均为浮点数
float_cols = [c for c in loan_stats.columns if ('应还本金' in c)] + ['放款金额', '金额_定价', '金额_期限']
float_cols = [c for c in dict.fromkeys(float_cols) if c in loan_stats.columns]  # 去重 + 存在性过滤
float_cols += [
    'mob1_累计复贷金额','mob2_累计复贷金额','mob3_累计复贷金额','mob4_累计复贷金额','mob5_累计复贷金额',
    'mob6_累计复贷金额','mob7_累计复贷金额','mob8_累计复贷金额','mob9_累计复贷金额','mob10_累计复贷金额',
    'mob11_累计复贷金额','mob12_累计复贷金额','mob13_累计复贷金额','mob14_累计复贷金额','mob15_累计复贷金额',
    'mob16_累计复贷金额','mob17_累计复贷金额','mob18_累计复贷金额',
    'mob1_vip_revenue','mob2_vip_revenue','mob3_vip_revenue','mob4_vip_revenue','mob5_vip_revenue',
    'mob6_vip_revenue','mob7_vip_revenue','mob8_vip_revenue','mob9_vip_revenue','mob10_vip_revenue',
    'mob11_vip_revenue','mob12_vip_revenue','mob13_vip_revenue','mob14_vip_revenue','mob15_vip_revenue',
    'mob16_vip_revenue','mob17_vip_revenue',
]
for col in float_cols:
    if col in loan_stats.columns:
        loan_stats[col] = pd.to_numeric(loan_stats[col], errors='coerce').astype('float64') # “清洗 + 转类型”，对混有字符串数字（如 "12.3"）、空字符串、非法字符等也能处理
        # loan_stats[col] = loan_stats[col].astype('float64')  # 不做解析，只做 dtype 强转，遇到 "1,234"、"—"、""、'abc' 这类会直接报 ValueError
        # loan_stats[col] = pd.to_numeric(loan_stats[col], errors='coerce').fillna(0).astype('float64') # 填充空值为零，但dataworks实际空值导出也是空白，非零值

# 整数字段
int_cols = [
    '放款人数',
    '出账1_1订单','出账1_2订单','出账1_3订单','出账1_4订单','出账1_7订单','出账1_15订单','出账1_30订单',
    '出账2_7订单','出账2_15订单','出账2_30订单',
    '出账3_7订单','出账3_15订单','出账3_30订单',
    '出账4_30订单','出账5_30订单','出账6_30订单','出账7_30订单','出账8_30订单','出账9_30订单',
    '出账10_30订单','出账11_30订单','出账12_30订单','出账13_30订单','出账14_30订单','出账15_30订单',
]
for col in int_cols:
    if col in loan_stats.columns:
        # loan_stats[col] = pd.to_numeric(loan_stats[col], errors='coerce').fillna(0).astype('Int64') 
        loan_stats[col] = pd.to_numeric(loan_stats[col], errors='coerce').astype('Int64')  # 使用Pandas 的 Int64（可空整数）/ NumPy 的 int64（普通整数）

file_path = r"D:\9.极限给额专题分析\极限给额LTV.xlsx"
write_dataframe_to_excel_com(file_path, {"分析数据": loan_stats}, start_row=1, include_header=True)